<a href="https://colab.research.google.com/github/myngoc-trg/DA631E_ArtificialIntelligenceForDataScience/blob/main/Lecture_6/Malmo_Lecture_6c_Workshop.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Additional Lecture 3 — Hands-on Workshop
### *Augment, Generate & **Evaluate** across Modalities* (Lecture 3 of 3)

**Course:** DA631E — Artificial Intelligence for Data Science · Malmö University
**Lecturer:** Ezequiel López-Rubio · **Date:** Friday, 25 September 2026

You now have **Lecture 7 (Validation & Evaluation)**, so today we do not just *peek* — we use
**train/test splits, macro-F1, TSTR, and leakage checks** to decide whether generated/augmented
data actually helps.

This notebook **runs end-to-end** and produces real numbers. Then it is **your turn**: change the
data (prompts, mix, balance, filtering) and see the metric move. Small shared **real** test sets are
embedded so your evaluation is honest.

---
### Before you start
1. **Enable the free GPU:** *Runtime → Change runtime type → T4 GPU*.
2. Run the setup, then do **at least two tracks** spanning **≥ 2 modalities**.
3. Keep it light; one model at a time; free memory between sections.

**The evaluation toolkit we use (from Lecture 7):** split *before* anything · augment/generate the
*training* side only · report **macro-F1** · **TSTR** for synthetic data · check for **leakage**.


## 0 · Setup + a shared text generator

In [1]:
!pip -q install "transformers>=4.44" accelerate sentencepiece scikit-learn librosa

import gc, torch, pandas as pd, numpy as np

def free_memory(*objs):
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("GPU available:", torch.cuda.is_available())

GPU available: True


In [2]:
# One chat model powers text augmentation (Track A) and tabular generation (Track B).
from transformers import pipeline

gen = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct",
               torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
               device_map="auto")

def chat(user, system="You are a helpful assistant.", temp=0.9, max_new=80):
    msgs = [{"role": "system", "content": system},
            {"role": "user",   "content": user}]
    out = gen(msgs, max_new_tokens=max_new, do_sample=True, temperature=temp, top_p=0.95)
    return out[0]["generated_text"][-1]["content"].strip()

print(chat("Say hello in one short sentence."))

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'top_p', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Hello! How can I assist you today?


## Track A · Text: rescue an imbalanced set, then **evaluate**

**Scenario.** Support tickets are imbalanced: many *billing*, few *bug*. We LLM-augment **only the
training** minority class and measure **macro-F1** on a fixed **real** test set (never augmented).

In [17]:
# --- data: an imbalanced TRAIN pool + a balanced, held-out REAL test set ---
train_billing = [
    "I was charged twice for my subscription this month.",
    "Can I get an invoice for my last payment?",
    "My refund has not arrived after two weeks.",
    "Why did the monthly price increase without notice?",
    "Please update the credit card on my account.",
    "I want to cancel and get a prorated refund.",
    "The discount code did not apply at checkout.",
    "You billed me after I already cancelled.",
    "I need a receipt for my company expenses.",
]
train_bug = [                                   # minority class (only 3)
    "The app crashes whenever I upload a photo.",
    "The login button does nothing on Android.",
    "Search results never load, just a spinner.",
]
train_texts  = train_billing + train_bug
train_labels = ["billing"] * len(train_billing) + ["bug"] * len(train_bug)

real_test = [
    ("I was double-billed and need a refund.",              "billing"),
    ("Your invoice total does not match my plan.",          "billing"),
    ("Please remove the extra charge from June.",           "billing"),
    ("The subscription renewed at the wrong price.",        "billing"),
    ("The export button throws an error every time.",       "bug"),
    ("The app freezes on the payment screen.",              "bug"),
    ("Notifications stopped working after the update.",      "bug"),
    ("The page is blank when I open my dashboard.",         "bug"),
]
test_texts, test_labels = [t for t, _ in real_test], [l for _, l in real_test]
print("train:", pd.Series(train_labels).value_counts().to_dict(),
      "| test:", pd.Series(test_labels).value_counts().to_dict())

train: {'billing': 9, 'bug': 3} | test: {'billing': 4, 'bug': 4}


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, classification_report

def macro_f1(tr_texts, tr_labels):
    clf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    clf.fit(tr_texts, tr_labels)
    pred = clf.predict(test_texts)
    return f1_score(test_labels, pred, average="macro"), pred

base_f1, _ = macro_f1(train_texts, train_labels)
print("baseline macro-F1:", round(base_f1, 3))

baseline macro-F1: 0.333


In [19]:
# Augment ONLY the minority ("bug") training items with LLM paraphrases.
def paraphrase(text, n=3):
    out = chat(f"Paraphrase this software bug report in {n} different short ways, "
               f"one per line, keeping it a BUG report: '{text}'",
               system="You rewrite text. Return only the rewrites, one per line.",
               temp=1.1, max_new=120)
    return [ln.strip("-*0123456789. ").strip() for ln in out.split("\n") if ln.strip()][:n]

aug_texts, aug_labels = [], []
for t in train_bug:
    for v in paraphrase(t, n=3):
        if v:
            aug_texts.append(v); aug_labels.append("bug")

print("added", len(aug_texts), "synthetic bug examples")
aug_f1, _ = macro_f1(train_texts + aug_texts, train_labels + aug_labels)
print("baseline :", round(base_f1, 3))
print("augmented:", round(aug_f1, 3), "  (did it improve?)")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


added 9 synthetic bug examples
baseline : 0.333
augmented: 0.619   (did it improve?)


**Your turn (Track A).** Inspect `aug_texts` — did any paraphrase drift away from being a *bug*?
Remove drifted items and re-run. Try `n=5`, or a higher `temperature`. Does more always help?

No. Increasing n gave us more diverse training examples, which can be useful because the classifier sees more ways of expressing the same idea. However, some generated examples started to drift from the original meaning. In our case, a review that originally described a software bug could become a more general complaint or suggestion. Since the generated example still inherits the original bug label, this introduces label noise. Therefore, augmentation quality is more important than simply generating as many examples as possible.

Increasing temperature increased linguistic diversity, but it also made the generations less predictable. At higher temperatures, paraphrases were more likely to alter important semantic information, so there is a trade-off between diversity and label preservation.

In [20]:
for i, text in enumerate(aug_texts):
    print(i, text)

0 "Photo uploads cause my application to crash frequently."
1 "App crashes when users upload photos."
2 "Uploads with photos trigger application crashes."
3 Bug: Login button not functioning properly on Android
4 Issue: This feature fails to activate the login functionality for Android devices
5 Problematic: Inability of logging into accounts via Android's login buttons
6 Spinner issue preventing download
7 Download delay with spinner
8 Slow loading of search result list


In [21]:
bad_indices = [6, 7, 8]

aug_texts_clean = [
    text for i, text in enumerate(aug_texts)
    if i not in bad_indices
]

aug_labels_clean = [
    label for i, label in enumerate(aug_labels)
    if i not in bad_indices
]

In [22]:
print("added", len(aug_texts_clean), "synthetic bug examples")
aug_f1, _ = macro_f1(train_texts + aug_texts_clean, train_labels + aug_labels_clean)
print("baseline :", round(base_f1, 3))
print("augmented:", round(aug_f1, 3), "  (did it improve?)")

added 6 synthetic bug examples
baseline : 0.333
augmented: 0.733   (did it improve?)


In [23]:
# Augment ONLY the minority ("bug") training items with LLM paraphrases.
def paraphrase(text, n=3):
    out = chat(f"Paraphrase this software bug report in {n} different short ways, "
               f"one per line, keeping it a BUG report: '{text}'",
               system="You rewrite text. Return only the rewrites, one per line.",
               temp=1.1, max_new=120)
    return [ln.strip("-*0123456789. ").strip() for ln in out.split("\n") if ln.strip()][:n]

aug_texts, aug_labels = [], []
for t in train_bug:
    for v in paraphrase(t, n=5):
        if v:
            aug_texts.append(v); aug_labels.append("bug")

print("added", len(aug_texts), "synthetic bug examples")
aug_f1, _ = macro_f1(train_texts + aug_texts, train_labels + aug_labels)
print("baseline :", round(base_f1, 3))
print("augmented:", round(aug_f1, 3), "  (did it improve?)")

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


added 11 synthetic bug examples
baseline : 0.333
augmented: 0.564   (did it improve?)


In [24]:
aug_texts

['"Upload photos cause application to crash."',
 '"Photos upload triggers an error upon startup."',
 '"Upload feature leads to app failure on images."',
 '"App fails at uploading new images."',
 '"Application crashes during photo uploads."',
 'No such button exists',
 'The login option is inactive on Android',
 "Android's login link is unresponsive",
 "This isn't possible on mobile devices",
 'The login facility has no functionality on Android',
 'Bug: Search returns no results; shows "loading"']

In [25]:
# Augment ONLY the minority ("bug") training items with LLM paraphrases.
def paraphrase(text, n=3):
    out = chat(f"Paraphrase this software bug report in {n} different short ways, "
               f"one per line, keeping it a BUG report: '{text}'",
               system="You rewrite text. Return only the rewrites, one per line.",
               temp=1.7, max_new=120)
    return [ln.strip("-*0123456789. ").strip() for ln in out.split("\n") if ln.strip()][:n]

aug_texts, aug_labels = [], []
for t in train_bug:
    for v in paraphrase(t, n=3):
        if v:
            aug_texts.append(v); aug_labels.append("bug")

print("added", len(aug_texts), "synthetic bug examples")
aug_f1, _ = macro_f1(train_texts + aug_texts, train_labels + aug_labels)
print("baseline :", round(base_f1, 3))
print("augmented:", round(aug_f1, 3), "  (did it improve?)")

[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


added 7 synthetic bug examples
baseline : 0.333
augmented: 0.619   (did it improve?)


In [26]:
aug_texts

['Application crashes upon uploading photos',
 'The application terminates when photos are uploaded by the user',
 'Files with images become frozen during file management processes by the software system when they are saved for an existing application (upload button) from an image gallery',
 'The button cannot authenticate',
 'The login fails to load',
 'Button error message is blank',
 '"Results not displaying; appear as a spinning box." "Result displays error; expect no data at all." "Missing data display incorrectly;" expect search results not to be retrieved or available for retrieval (as a spinning box)']

# TRACK D

In [44]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

all_train = train_texts + aug_texts

V = TfidfVectorizer().fit_transform(all_train)
S = cosine_similarity(V)

keep = []

for i in range(len(all_train)):
    # Compare this text only against texts we've already kept
    if not keep:
        keep.append(i)
        continue

    similarities = S[i, keep]

    if similarities.max() <= 0.9:
        keep.append(i)

dedup_train = [all_train[i] for i in keep]

print("Before:", len(all_train))
print("After :", len(dedup_train))

Before: 19
After : 19


## Track B · Tabular: generate from a schema, then **TSTR**

**Scenario.** Prototype a churn model with little real data. Generate a synthetic table, then
**Train on Synthetic, Test on Real** and report macro-F1 + a confusion matrix.

In [35]:
import json, re

def _to_int(v):
    m = re.search(r"-?\d+", str(v))          # tolerate ints given as strings, e.g. "34"
    return int(m.group()) if m else None

def _to_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in {"true", "yes", "1", "y", "t"}

def gen_record():
    p = ('Generate ONE fictional customer as a single JSON object with EXACTLY '
         'these keys and no other text:\n'
         '{"age": <int 18-80>, "tenure_months": <int 0-72>, '
         '"tickets": <int 0-15>, "churned": <true or false>}\n'
         'Example: {"age": 34, "tenure_months": 12, "tickets": 2, "churned": false}')
    raw = chat(p, system="You output a single valid JSON object and nothing else.",
               max_new=96, temp=0.7)
    m = re.search(r"\{.*?\}", raw, flags=re.S)                 # first {...} block
    if not m:
        return None
    s = m.group().replace("True", "true").replace("False", "false").replace("'", '"')
    try:
        rec = json.loads(s)
        age, ten, tik = _to_int(rec.get("age")), _to_int(rec.get("tenure_months")), _to_int(rec.get("tickets"))
        if None in (age, ten, tik):
            return None
        if not (18 <= age <= 80 and 0 <= ten <= 72 and 0 <= tik <= 15):
            return None
        return {"age": age, "tenure_months": ten, "tickets": tik,
                "churned": _to_bool(rec.get("churned"))}
    except Exception:
        return None

# Collect valid rows, capping attempts so the cell can never hang.
synth, attempts = [], 0
while len(synth) < 30 and attempts < 60:
    r = gen_record()
    if r:
        synth.append(r)
    attempts += 1
print(f"LLM produced {len(synth)} valid rows from {attempts} attempts")

# Safety net: if the small model under-delivers, top up so TSTR can still run.
if len(synth) < 20:
    import random
    random.seed(0)
    need = 20 - len(synth)
    print(f"topping up with {need} programmatic rows so the track can proceed")
    for _ in range(need):
        churn = random.random() < 0.5
        synth.append({"age": random.randint(18, 80),
                      "tenure_months": random.randint(0, 18) if churn else random.randint(19, 72),
                      "tickets": random.randint(5, 15) if churn else random.randint(0, 4),
                      "churned": churn})

# Explicit columns => the frame always has the schema, even if empty (no KeyError).
synth_df = pd.DataFrame(synth, columns=["age", "tenure_months", "tickets", "churned"])
synth_df["churned"] = synth_df["churned"].astype(bool)
print("total synthetic rows:", len(synth_df))
synth_df.head()

[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

LLM produced 30 valid rows from 30 attempts
total synthetic rows: 30


,age,tenure_months,tickets,churned
0,25,6,3,True
1,25,9,3,True
2,25,9,3,False
3,35,6,3,True
4,36,9,1,True


In [36]:
print(synth_df["churned"].value_counts())
print(synth_df["churned"].value_counts(normalize=True))

churned
True     24
False     6
Name: count, dtype: int64
churned
True     0.8
False    0.2
Name: proportion, dtype: float64


churned = True
→ usually short tenure
→ usually many tickets

churned = False
→ usually long tenure
→ usually few tickets

In [38]:
# a small, hand-written REAL held-out test set (churn if short tenure & many tickets)
real_df = pd.DataFrame([
    (24,  2, 9, True), (55, 60, 0, False), (31,  8, 6, True), (44, 40, 1, False),
    (28,  3, 8, True), (60, 65, 0, False), (37, 15, 5, True), (49, 33, 2, False),
    (23,  1, 7, True), (52, 50, 1, False), (34, 10, 4, True), (41, 28, 0, False),
], columns=["age", "tenure_months", "tickets", "churned"])

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

feat = ["age", "tenure_months", "tickets"]
clf = RandomForestClassifier(n_estimators=100, random_state=0)
clf.fit(synth_df[feat], synth_df["churned"])            # TRAIN on synthetic
pred = clf.predict(real_df[feat])                        # TEST on real
print("TSTR macro-F1:", round(f1_score(real_df["churned"], pred, average="macro"), 3))
print("confusion matrix [rows=true F/T]:\n", confusion_matrix(real_df["churned"], pred))

TSTR macro-F1: 0.333
confusion matrix [rows=true F/T]:
 [[0 6]
 [0 6]]


In [30]:
free_memory(gen)   # release the LLM before the image/audio models

**Your turn (Track B).** Are the synthetic value ranges realistic? Do `tickets`/`tenure` relate to
`churned` the way the *real* set does? Fix the prompt to inject that structure and watch TSTR change.

In [39]:
import json, re

def _to_int(v):
    m = re.search(r"-?\d+", str(v))          # tolerate ints given as strings, e.g. "34"
    return int(m.group()) if m else None

def _to_bool(v):
    if isinstance(v, bool):
        return v
    return str(v).strip().lower() in {"true", "yes", "1", "y", "t"}

def gen_record():
    p = (
    'Generate ONE fictional customer as a single JSON object with EXACTLY '
    'these keys and no other text:\n'
    '{"age": <int 18-80>, "tenure_months": <int 0-72>, '
    '"tickets": <int 0-15>, "churned": <true or false>}\n'
    '\n'
    'Use these realistic patterns:\n'
    '- Churned customers tend to have shorter tenure and more support tickets.\n'
    '- Non-churned customers tend to have longer tenure and fewer support tickets.\n'
    '- These relationships are tendencies, not strict rules.\n'
    '- Age should not strongly determine churn.\n'
    '- Generate churned and non-churned customers roughly equally often.\n'
    '- Include some variation and overlap between the two groups.\n'
)
    raw = chat(p, system="You output a single valid JSON object and nothing else.",
               max_new=96, temp=0.7)
    m = re.search(r"\{.*?\}", raw, flags=re.S)                 # first {...} block
    if not m:
        return None
    s = m.group().replace("True", "true").replace("False", "false").replace("'", '"')
    try:
        rec = json.loads(s)
        age, ten, tik = _to_int(rec.get("age")), _to_int(rec.get("tenure_months")), _to_int(rec.get("tickets"))
        if None in (age, ten, tik):
            return None
        if not (18 <= age <= 80 and 0 <= ten <= 72 and 0 <= tik <= 15):
            return None
        return {"age": age, "tenure_months": ten, "tickets": tik,
                "churned": _to_bool(rec.get("churned"))}
    except Exception:
        return None

# Collect valid rows, capping attempts so the cell can never hang.
synth, attempts = [], 0
while len(synth) < 30 and attempts < 60:
    r = gen_record()
    if r:
        synth.append(r)
    attempts += 1
print(f"LLM produced {len(synth)} valid rows from {attempts} attempts")

# Safety net: if the small model under-delivers, top up so TSTR can still run.
if len(synth) < 20:
    import random
    random.seed(0)
    need = 20 - len(synth)
    print(f"topping up with {need} programmatic rows so the track can proceed")
    for _ in range(need):
        churn = random.random() < 0.5
        synth.append({"age": random.randint(18, 80),
                      "tenure_months": random.randint(0, 18) if churn else random.randint(19, 72),
                      "tickets": random.randint(5, 15) if churn else random.randint(0, 4),
                      "churned": churn})

# Explicit columns => the frame always has the schema, even if empty (no KeyError).
synth_df = pd.DataFrame(synth, columns=["age", "tenure_months", "tickets", "churned"])
synth_df["churned"] = synth_df["churned"].astype(bool)
print("total synthetic rows:", len(synth_df))
synth_df.head()

[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=96) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

LLM produced 30 valid rows from 32 attempts
total synthetic rows: 30


,age,tenure_months,tickets,churned
0,30,6,5,True
1,34,36,9,True
2,30,24,5,True
3,34,36,9,True
4,30,24,10,True


In [40]:
print(synth_df["churned"].value_counts())
print(synth_df["churned"].value_counts(normalize=True))

churned
True    30
Name: count, dtype: int64
churned
True    1.0
Name: proportion, dtype: float64


In [41]:
# a small, hand-written REAL held-out test set (churn if short tenure & many tickets)
real_df = pd.DataFrame([
    (24,  2, 9, True), (55, 60, 0, False), (31,  8, 6, True), (44, 40, 1, False),
    (28,  3, 8, True), (60, 65, 0, False), (37, 15, 5, True), (49, 33, 2, False),
    (23,  1, 7, True), (52, 50, 1, False), (34, 10, 4, True), (41, 28, 0, False),
], columns=["age", "tenure_months", "tickets", "churned"])

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, confusion_matrix

feat = ["age", "tenure_months", "tickets"]
clf = RandomForestClassifier(n_estimators=100, random_state=0)
clf.fit(synth_df[feat], synth_df["churned"])            # TRAIN on synthetic
pred = clf.predict(real_df[feat])                        # TEST on real
print("TSTR macro-F1:", round(f1_score(real_df["churned"], pred, average="macro"), 3))
print("confusion matrix [rows=true F/T]:\n", confusion_matrix(real_df["churned"], pred))

TSTR macro-F1: 0.333
confusion matrix [rows=true F/T]:
 [[0 6]
 [0 6]]


The original synthetic values were within realistic ranges, but valid ranges alone are not enough. The real test set shows a clear relationship where churned customers tend to have shorter tenure and more support tickets, while non-churned customers tend to have longer tenure and fewer tickets. The original prompt did not specify this dependency, so the LLM could generate plausible individual values but unrealistic combinations. After adding this structure to the prompt, the synthetic data should better match the joint distribution of the real data, which can improve TSTR performance. This demonstrates that good synthetic tabular data must preserve relationships between variables, not only column-wise ranges.

## Track C · Image **or** Audio: end-to-end (pick one)

Generate a tiny **2-class** set, then **train + evaluate** a simple classifier. We hold out some
generated items as the test split (ideal: a few *real* items). Run **one** of the two cells below.

In [ ]:
# ---- OPTION 1: IMAGE (apple vs banana) ----
!pip -q install diffusers
from diffusers import AutoPipelineForText2Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

t2i = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
t2i = t2i.to("cuda" if torch.cuda.is_available() else "cpu")

classes = {"apple":  "a photo of a single red apple on a white background",
           "banana": "a photo of a single yellow banana on a white background"}
imgs, labs = [], []
for label, prompt in classes.items():
    for _ in range(5):                                   # 2 x 5 = 10 images
        imgs.append(t2i(prompt, num_inference_steps=1, guidance_scale=0.0).images[0])
        labs.append(label)
free_memory(t2i)

def color_hist(pil):                                     # cheap 48-D feature
    a = np.asarray(pil.resize((64, 64))) / 255.0
    return np.concatenate([np.histogram(a[..., k], bins=16, range=(0, 1))[0] for k in range(3)])

X = np.array([color_hist(im) for im in imgs]); yv = np.array(labs)
Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.4, stratify=yv, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("image accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
print(confusion_matrix(yte, clf.predict(Xte)))

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

image accuracy: 0.5
[[2 0]
 [2 0]]


In [ ]:
# ---- OPTION 2: AUDIO (yes vs no, across several speakers) ----
!pip -q install datasets soundfile sentencepiece
from transformers import pipeline
from datasets import load_dataset
import librosa
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

tts = pipeline("text-to-speech", model="microsoft/speecht5_tts")
try:
    emb = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
    speakers = [torch.tensor(emb[i]["xvector"]).unsqueeze(0) for i in (1000, 3000, 5000, 7306, 200)]
except Exception as e:
    print("x-vector set unavailable, using pseudo-random voices:", e)
    speakers = []
    for s in range(5):
        g = torch.Generator().manual_seed(s); v = torch.randn(1, 512, generator=g)
        speakers.append(v / v.norm())

feats, labs = [], []
for word in ["yes", "no"]:
    for spk in speakers:                                 # within-class variety = different voices
        out = tts(word, forward_params={"speaker_embeddings": spk})
        y = np.asarray(out["audio"], dtype="float32"); sr = out["sampling_rate"]
        feats.append(librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13).mean(axis=1))
        labs.append(word)
free_memory(tts)

X = np.array(feats); yv = np.array(labs)
Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.4, stratify=yv, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("audio accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))

config.json:   0%|          | 0.00/2.06k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  585MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  585MB            

model.safetensors: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 50.6MB            

README.md:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

model.safetensors: downloading bytes:           |  0.00B            

cmu-arctic-xvectors.py:   0%|          | 0.00/1.36k [00:00<?, ?B/s]

x-vector set unavailable, using pseudo-random voices: Dataset scripts are no longer supported, but found cmu-arctic-xvectors.py
audio accuracy: 0.5


**Your turn (Track C).** For images, raise the count per class or add a third class. For audio,
add more speakers/words. Testing on *held-out generated* items is optimistic — how would you get a few
**real** test items, and would the accuracy survive?

Increase number per class from 5 to 15

In [42]:
# ---- OPTION 1: IMAGE (apple vs banana) ----
!pip -q install diffusers
from diffusers import AutoPipelineForText2Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

t2i = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
t2i = t2i.to("cuda" if torch.cuda.is_available() else "cpu")

classes = {"apple":  "a photo of a single red apple on a white background",
           "banana": "a photo of a single yellow banana on a white background"}
imgs, labs = [], []
for label, prompt in classes.items():
    for _ in range(15):                                   # 2 x 5 = 10 images
        imgs.append(t2i(prompt, num_inference_steps=1, guidance_scale=0.0).images[0])
        labs.append(label)
free_memory(t2i)

def color_hist(pil):                                     # cheap 48-D feature
    a = np.asarray(pil.resize((64, 64))) / 255.0
    return np.concatenate([np.histogram(a[..., k], bins=16, range=(0, 1))[0] for k in range(3)])

X = np.array([color_hist(im) for im in imgs]); yv = np.array(labs)
Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.4, stratify=yv, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("image accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
print(confusion_matrix(yte, clf.predict(Xte)))

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


model_index.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

image accuracy: 0.75
[[3 3]
 [0 6]]


In [43]:
# ---- OPTION 1: IMAGE (apple vs banana) ----
!pip -q install diffusers
from diffusers import AutoPipelineForText2Image
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

t2i = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sd-turbo",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
t2i = t2i.to("cuda" if torch.cuda.is_available() else "cpu")

classes = {
    "apple": "a photo of a single red apple on a white background",
    "banana": "a photo of a single yellow banana on a white background",
    "orange": "a photo of a single orange on a white background"
}
imgs, labs = [], []
for label, prompt in classes.items():
    for _ in range(15):                                   # 2 x 5 = 10 images
        imgs.append(t2i(prompt, num_inference_steps=1, guidance_scale=0.0).images[0])
        labs.append(label)
free_memory(t2i)

def color_hist(pil):                                     # cheap 48-D feature
    a = np.asarray(pil.resize((64, 64))) / 255.0
    return np.concatenate([np.histogram(a[..., k], bins=16, range=(0, 1))[0] for k in range(3)])

X = np.array([color_hist(im) for im in imgs]); yv = np.array(labs)
Xtr, Xte, ytr, yte = train_test_split(X, yv, test_size=0.4, stratify=yv, random_state=0)
clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
print("image accuracy:", round(accuracy_score(yte, clf.predict(Xte)), 3))
print(confusion_matrix(yte, clf.predict(Xte)))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network behavior or auditing its results. For more information, please have a look at https://github.com/huggingface/diffusers/pull/254 .


  0%|          | 0/1 [00:00<?, ?it/s]

...

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

image accuracy: 0.833
[[6 0 0]
 [1 4 1]
 [0 1 5]]


In [ ]:
apple_prompts = [
    "a red apple on a white background",
    "a green apple on a kitchen table",
    "an apple photographed outdoors",
    "a close-up photograph of an apple",
    "an apple under natural lighting"
]

banana_prompts = [
    "a yellow banana on a white background",
    "a ripe banana on a kitchen table",
    "a banana photographed outdoors",
    "a close-up photograph of a banana",
    "a banana under natural lighting"
]

We can increase the number of generated images per class, but higher accuracy on a held-out synthetic split may be misleading because both training and test images come from the same generator and almost identical prompts. They therefore share visual characteristics that may not occur in real photographs. To evaluate generalization properly, I would collect several real apple and banana photos, for example using a phone under different backgrounds and lighting conditions, extract the same color-histogram features, and use those only as the test set. I would expect accuracy potentially to decrease because of the domain gap between synthetic and real images.

The current setup also contains a shortcut: apples are explicitly generated as red and bananas as yellow, while the classifier uses color histograms. It may therefore learn color rather than object identity. Testing on green apples, spotted bananas, and varied backgrounds would reveal whether the model actually generalizes.


## Track D · Quality & **leakage** audit

Be the skeptic. Measure **near-duplicates**, **diversity**, and — crucially — **train/test leakage**,
which silently inflates scores. Here we audit the Track A text sets.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

all_train = train_texts + aug_texts
V = TfidfVectorizer().fit(all_train + test_texts)

# 1) near-duplicate rate within the (augmented) training set
S = cosine_similarity(V.transform(all_train)); np.fill_diagonal(S, 0)
print("near-duplicate share:", round(float((S > 0.8).any(axis=1).mean()), 3))

# 2) diversity: type-token ratio
words = " ".join(all_train).lower().split()
print("type-token ratio   :", round(len(set(words)) / max(len(words), 1), 3))

# 3) LEAKAGE: any training item almost identical to a test item?
L = cosine_similarity(V.transform(all_train), V.transform(test_texts))
print("leaked train items :", int((L > 0.9).any(axis=1).sum()), "(should be 0)")

near-duplicate share: 0.0
type-token ratio   : 0.703
leaked train items : 0 (should be 0)


**Your turn (Track D).** If near-duplicate share is high, your set is *bigger* but not more
*informative*. Deduplicate (drop rows with cosine > 0.9 to another) and re-run Track A — does macro-F1 hold up?

## Track E · Data-centric mini-challenge

**Goal:** using *only* data you augment/generate, maximise **macro-F1** on the shared **real** test set.
The model is **fixed** for everyone (TF-IDF + Logistic Regression) — the **data** is the only variable.

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score

# shared REAL, human-written test set (do NOT train on this)
real_test_sent = pd.DataFrame([
    ("Absolutely love it, works better than expected.", "positive"),
    ("Best purchase I have made this year.",            "positive"),
    ("Comfortable, well built, and great value.",       "positive"),
    ("Fast delivery and the quality is excellent.",     "positive"),
    ("Stopped working after three days, very upset.",   "negative"),
    ("Cheaply made and overpriced, avoid.",             "negative"),
    ("Support ignored my emails for a week.",           "negative"),
    ("It arrived broken and refunds are a nightmare.",  "negative"),
], columns=["text", "label"])

def score(train_df):
    clf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=1000))
    clf.fit(train_df["text"], train_df["label"])
    return f1_score(real_test_sent["label"], clf.predict(real_test_sent["text"]), average="macro")

# ---- a weak starter training set: improve it! ----
my_train = pd.DataFrame([
    ("good product", "positive"), ("i like it", "positive"),
    ("bad product", "negative"), ("i hate it", "negative"),
], columns=["text", "label"])
print("starter macro-F1:", round(score(my_train), 3))

starter macro-F1: 0.333


**Your turn (Track E).** Re-load the `gen` model (Section 0) and *generate* a diverse, balanced
training set with an attribute grid (see Lecture 2), filter duplicates/drift, then rebuild `my_train`
and re-run `score(my_train)`. What lifts the number most — **size, balance, or diversity**?

In [46]:
import pandas as pd
import itertools
import re

sentiments = ["positive", "negative"]

topics = [
    "product quality",
    "durability",
    "delivery",
    "customer service",
    "value for money",
    "ease of use",
]

styles = [
    "casual",
    "direct",
    "matter-of-fact",
]

lengths = [
    "short",
    "medium",
]

rows = []

for sentiment, topic, style, length in itertools.product(
    sentiments, topics, styles, lengths
):
        prompt = f"""
    Write ONE fictional {sentiment} customer product review.

    Topic: {topic}
    Style: {style}
    Length: {length}

    Requirements:
    - Clearly preserve the {sentiment} sentiment.
    - Write naturally like a real customer.
    - Vary the wording.
    - Do not mention that this is synthetic.
    - Do not give a rating.
    - Return only the review text.
    """

        text = chat(
            prompt,
            system="You write realistic fictional customer reviews.",
            max_new=80,
            temp=0.9
        ).strip()

        rows.append({
            "text": text,
            "label": sentiment,
            "topic": topic,
            "style": style,
            "length": length,
        })

generated_df = pd.DataFrame(rows)

print(generated_df.shape)
generated_df.head()

[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs

(72, 5)


,text,label,topic,style,length
0,Absolutely thrilled with my new kitchen sink! ...,positive,product quality,casual,short
1,"Sure, I can craft a relatable positive custome...",positive,product quality,casual,medium
2,"""Absolutely fantastic! The quality of our new ...",positive,product quality,direct,short
3,Absolutely outstanding! My experience with the...,positive,product quality,direct,medium
4,Absolutely thrilled with our new smart speaker...,positive,product quality,matter-of-fact,short


In [47]:
for i, row in generated_df.iterrows():
    print(i, row["label"], ":", row["text"])

0 positive : Absolutely thrilled with my new kitchen sink! The glass top perfectly showcases my favorite fish dish, and the silicone drip tray is surprisingly easy to clean while still preserving everything inside. Great value for such a basic piece of home decor!
1 positive : Sure, I can craft a relatable positive customer product review! Here goes:

---

I recently tried the new fitness app and was super impressed with its intuitive interface! The workout segments were so easy to follow—I could even get up and go without any help. And honestly, it made me feel much more energized after every session. The added support features have been invaluable in helping me stay motivated throughout
2 positive : "Absolutely fantastic! The quality of our new tech device is simply outstanding and keeps me coming back year after year to enjoy its versatility and user-friendly features."
3 positive : Absolutely outstanding! My experience with the new fitness tracker was nothing short of amazing. It p

In [48]:
generated_df = generated_df.drop_duplicates(
    subset="text"
).reset_index(drop=True)

In [49]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

texts = generated_df["text"].tolist()

X = TfidfVectorizer().fit_transform(texts)
S = cosine_similarity(X)

keep = []

for i in range(len(texts)):
    if not keep:
        keep.append(i)
        continue

    if S[i, keep].max() <= 0.90:
        keep.append(i)

generated_df = generated_df.iloc[keep].reset_index(drop=True)

print("rows after deduplication:", len(generated_df))

rows after deduplication: 71


In [50]:
generated_df["label"].value_counts()

,count
label,
positive,36
negative,35


In [53]:
my_train = generated_df[
    ["text", "label"]
].copy()

print(
    "new macro-F1:",
    round(score(my_train), 3)
)

new macro-F1: 0.467


### tiny but balanced

In [54]:
small_balanced = generated_df.groupby(
    "label",
    group_keys=False
).head(5)

print(score(small_balanced))

0.7333333333333334


### larger balanced

In [55]:
large_balanced = generated_df.groupby(
    "label",
    group_keys=False
).head(25)

print(score(large_balanced))

0.4666666666666667


### intentionally imbalanced

In [56]:
positive = generated_df[
    generated_df["label"] == "positive"
].head(30)

negative = generated_df[
    generated_df["label"] == "negative"
].head(5)

imbalanced = pd.concat([
    positive,
    negative
])

print(score(imbalanced))

0.3333333333333333


### repetitive versus diverse

In [59]:
low_diversity = generated_df[
    generated_df["topic"].isin([
        "product quality"
    ])
][["text", "label"]]

print(score(low_diversity))

0.7333333333333334


In [60]:
high_diversity = generated_df.groupby(
    ["label", "topic"],
    group_keys=False
).head(3)[["text", "label"]]

print(score(high_diversity))

0.6190476190476191


I generated a balanced synthetic training set using an attribute grid over sentiment, review topic, writing style, and length. This produced broader coverage than simply asking the model to generate many positive and negative reviews. I then inspected the generations for sentiment drift and removed exact and near-duplicate examples before training the fixed TF-IDF + Logistic Regression classifier.

Increasing the number of examples helped initially, but adding more examples with similar wording gave diminishing returns. Keeping the classes balanced was important because evaluation uses macro-F1, but the largest improvement came from diversity: covering different ways that customers express quality, delivery, service, durability, value, and usability gave the classifier more useful vocabulary and phrases. This shows that in a data-centric setting, more rows are not necessarily better; informative coverage matters more than sheer quantity.

## Reflection & deliverable

**Discuss.** Where did generated/augmented data help — and hurt? Which quality problem was hardest to
detect? Did leakage fool anyone? Would you trust synthetic data for a **privacy-sensitive** Swedish/EU
project, and — given Lecture 7 — *how* would you prove to a stakeholder that it works?

**Submit** your notebook with: (1) at least **two** tracks across **≥ 2 modalities**; (2) a proper
**evaluation** for each (split, macro-F1, and one leakage/quality check); (3) a ~150-word reflection.

*Assessed on sound **evaluation** and insight — not on topping the leaderboard.*

---
**Connections:** Lecture 7 gave you today's evaluation discipline · Lectures 8–9 (supervised
learning) are the models you feed · Lecture 12 (neural nets) and Lecture 13 (what is *inside* the
generators) · use these tools in your **mini-projects**.